In [3]:
import csv
from collections import defaultdict
 
 
def load_csv(path):
    """Just reads the CSV into a header + list of rows, skipping blanks."""
    with open(path, newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        rows = [row for row in reader if row]
    return header, rows
 
 
def ask_for_the_new_record(header, rows):
    """
    Walks the user through entering a value for every column
    (except the last one, since that's what we're predicting).
 
    For employee_data.csv, this is where you'd type in things like
    the Gender, Department, Designation, and City of the person
    you want to classify.
    """
    columns_to_ask_about = header[:-1]
    new_record = {}
 
    print("\nOkay, let's build the record you want to classify.")
    print("-" * 60)
    for i, column in enumerate(columns_to_ask_about):
        seen_values = sorted(set(row[i] for row in rows))
        print(f"{column} - here's what shows up in the data: {seen_values}")
        answer = input(f"  What's the {column}? ").strip()
        new_record[column] = answer
 
    return new_record
 
 
def naive_bayes(header, rows, new_record):
    columns_to_ask_about = header[:-1]
    target_column = header[-1]  # e.g. "Remote Eligible"
    total_rows = len(rows)
 
    print("\n" + "=" * 60)
    print(f"Columns we're using: {columns_to_ask_about}")
    print(f"What we're predicting: '{target_column}'")
    print("=" * 60)
 
    # First, split everyone up by their class.
    # For our employee example: who's Remote Eligible, who isn't.
 
    grouped_by_class = defaultdict(list)
    for row in rows:
        grouped_by_class[row[-1]].append(row)
 
    print("Step 1: How many rows fall into each class")
    for label, group in grouped_by_class.items():
        print(f"  {target_column} = {label}: {len(group)} rows")
    print("=" * 60)
 
    # Priors: before looking at any details, what's the base rate
    # for each class? E.g. what fraction of employees are Remote
    # Eligible = Yes, just going by the raw counts?

    priors = {label: len(group) / total_rows
              for label, group in grouped_by_class.items()}
 
    print("Step 2: Prior probabilities (the base rates)")
    for label, p in priors.items():
        n = len(grouped_by_class[label])
        print(f"  P({target_column}={label}) = {n}/{total_rows} = {p:.4f}")
    print("=" * 60)
 

    # Now, for each column, how likely is the value the user gave us,
    # within each class? If we've never seen that combination before,
    # we smooth it a bit instead of just calling it impossible.
    # --------------------------------------------------------------
    print("Step 3: How likely is each answer, within each class?")
    print(f"The record we're classifying: {new_record}")
    print("-" * 60)
 
    likelihood_pieces = {label: [] for label in grouped_by_class}
    for column in columns_to_ask_about:
        col_index = header.index(column)
        answer = new_record[column]
        for label, group in grouped_by_class.items():
            matches = sum(1 for row in group if row[col_index] == answer)
            group_size = len(group)
            if matches == 0:
                # never seen this exact value in this class before,
                # so we lean on Laplace smoothing instead of a hard zero
                unique_values = len(set(row[col_index] for row in rows))
                probability = 1 / (group_size + unique_values)
                note = "  (smoothed - never saw this value in this class)"
            else:
                probability = matches / group_size
                note = ""
            likelihood_pieces[label].append(probability)
            print(f"  P({column}={answer}|{target_column}={label}) = "
                  f"{matches}/{group_size} = {probability:.4f}{note}")
    print("=" * 60)
 
    # --------------------------------------------------------------
    # The "naive" part: we pretend all the columns are independent
    # of each other and just multiply their probabilities together.
    # --------------------------------------------------------------
    print("Step 4: Multiply those probabilities together (the naive part)")
    combined_likelihood = {}
    for label, probabilities in likelihood_pieces.items():
        product = 1.0
        for p in probabilities:
            product *= p
        combined_likelihood[label] = product
        chain = " * ".join(f"{p:.4f}" for p in probabilities)
        print(f"  P(record|{target_column}={label}) = {chain} = {product:.6f}")
    print("=" * 60)
 
    # --------------------------------------------------------------
    # Combine each class's likelihood with its prior to get a score.
    # --------------------------------------------------------------
    print("Step 5: Fold in the prior for each class")
    scores = {}
    for label in grouped_by_class:
        scores[label] = combined_likelihood[label] * priors[label]
        print(f"  {combined_likelihood[label]:.6f} * {priors[label]:.4f} "
              f"= {scores[label]:.6f}")
    print("=" * 60)
 
    # --------------------------------------------------------------
    # Turn the raw scores into proper probabilities that add up to 1.
    # --------------------------------------------------------------
    total_score = sum(scores.values())
    print(f"Step 6: Normalize so everything adds up to 1 "
          f"({' + '.join(f'{s:.6f}' for s in scores.values())} = {total_score:.6f})")
    for label, s in scores.items():
        posterior = s / total_score
        print(f"  P({target_column}={label}|record) = {s:.6f}/{total_score:.6f} "
              f"= {posterior:.4f}")
    print("=" * 60)
 
    # --------------------------------------------------------------
    # Whichever class scored highest wins.
    # --------------------------------------------------------------
    winner = max(scores, key=scores.get)
    print(f"Step 7: And the winner is... {target_column} = {winner}")
    return winner
 
 
if __name__ == "__main__":
    csv_path = input("Path to your CSV (e.g. employee_data.csv): ").strip()
    header, rows = load_csv(csv_path)
    print(f"\nLoaded {len(rows)} rows from '{csv_path}'.")
    new_record = ask_for_the_new_record(header, rows)
    naive_bayes(header, rows, new_record)
 

Path to your CSV (e.g. employee_data.csv):  sample_data.csv



Loaded 15 rows from 'sample_data.csv'.

Okay, let's build the record you want to classify.
------------------------------------------------------------
id - here's what shows up in the data: ['1', '10', '11', '12', '13', '14', '15', '2', '3', '4', '5', '6', '7', '8', '9']


  What's the id?  


name - here's what shows up in the data: ['Aarav Sharma', 'Ananya Iyer', 'Arjun Kapoor', 'Dev Malhotra', 'Ishita Nair', 'Kabir Singh', 'Meera Joshi', 'Nikhil Verma', 'Priya Patel', 'Riya Bose', 'Rohan Mehta', 'Sameer Gupta', 'Sneha Reddy', 'Tara Menon', 'Vikram Rao']


  What's the name?  Rohan Mehta


...

join_date - here's what shows up in the data: ['2017-09-09', '2018-04-30', '2019-08-14', '2019-11-23', '2020-01-25', '2020-06-05', '2020-10-22', '2021-05-17', '2021-07-01', '2021-09-19', '2022-03-15', '2022-12-01', '2023-01-10', '2023-02-28', '2023-04-05']


  What's the join_date?  2023-02-28


rating - here's what shows up in the data: ['3.4', '3.5', '3.6', '3.7', '3.8', '3.9', '4.0', '4.1', '4.2', '4.3', '4.4', '4.5', '4.6', '4.8', '4.9']


  What's the rating?  3.7



Columns we're using: ['id', 'name', 'age', 'city', 'department', 'salary', 'join_date', 'rating']
What we're predicting: 'is_active'
Step 1: How many rows fall into each class
  is_active = Yes: 9 rows
  is_active = No: 6 rows
Step 2: Prior probabilities (the base rates)
  P(is_active=Yes) = 9/15 = 0.6000
  P(is_active=No) = 6/15 = 0.4000
Step 3: How likely is each answer, within each class?
The record we're classifying: {'id': '', 'name': 'Rohan Mehta', 'age': '22', 'city': 'Pune', 'department': 'HR', 'salary': '57000', 'join_date': '2023-02-28', 'rating': '3.7'}
------------------------------------------------------------
  P(id=|is_active=Yes) = 0/9 = 0.0417  (smoothed - never saw this value in this class)
  P(id=|is_active=No) = 0/6 = 0.0476  (smoothed - never saw this value in this class)
  P(name=Rohan Mehta|is_active=Yes) = 1/9 = 0.1111
  P(name=Rohan Mehta|is_active=No) = 0/6 = 0.0476  (smoothed - never saw this value in this class)
  P(age=22|is_active=Yes) = 1/9 = 0.1111
  P